In [1]:
# ==============================
# 1. Import Libraries
# ==============================
import re
import string
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import words

In [11]:
# Download required resources (run once)
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt_tab to /home/sagemaker-
[nltk_data]     user/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /home/sagemaker-
[nltk_data]     user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/sagemaker-
[nltk_data]     user/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/sagemaker-
[nltk_data]     user/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
# ==============================
# 2. Sample Dataset
# ==============================
text = """
OMG!!! This product is sooo goooood 😍😍!!! 
I bought it from https://example.com on 12/12/2024.
Contact me at test@email.com or call 9876543210.
#Amazing #BestProduct Ever!!!
"""

In [4]:
# ==============================
# 3. Basic Cleaning Functions
# ==============================

def to_lowercase(text):
    return text.lower()

def remove_urls(text):
    return re.sub(r'http\S+|www\S+', '', text)

def remove_emails(text):
    return re.sub(r'\S+@\S+', '', text)

def remove_phone_numbers(text):
    return re.sub(r'\b\d{10}\b', '', text)

def remove_html_tags(text):
    return re.sub(r'<.*?>', '', text)

def remove_emojis(text):
    return re.sub(r'[^\x00-\x7F]+', '', text)

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

def remove_numbers(text):
    return re.sub(r'\d+', '', text)

def remove_extra_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

In [5]:
# ==============================
# 4. Advanced Cleaning
# ==============================

# Expand contractions
contractions_dict = {
    "don't": "do not",
    "can't": "cannot",
    "i'm": "i am",
    "it's": "it is"
}

def expand_contractions(text):
    for key in contractions_dict:
        text = text.replace(key, contractions_dict[key])
    return text

# Handle repeated characters (sooo → soo)
def reduce_lengthening(text):
    return re.sub(r'(.)\1+', r'\1\1', text)

# Remove hashtags & mentions
def remove_hashtags_mentions(text):
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'@\w+', '', text)
    return text

In [6]:
# ==============================
# 5. Tokenization
# ==============================

def tokenize_text(text):
    words = word_tokenize(text)
    sentences = sent_tokenize(text)
    return words, sentences

In [7]:
# ==============================
# 6. Stopword Removal
# ==============================

stop_words = set(stopwords.words('english'))

def remove_stopwords(words):
    return [word for word in words if word not in stop_words]


In [8]:
# ==============================
# 7. Stemming & Lemmatization
# ==============================

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def apply_stemming(words):
    return [stemmer.stem(word) for word in words]

def apply_lemmatization(words):
    return [lemmatizer.lemmatize(word) for word in words]


In [9]:
# ==============================
# 8. Full Preprocessing Pipeline
# ==============================

def preprocess_text(text):
    print("Original Text:\n", text)

    text = to_lowercase(text)
    text = expand_contractions(text)
    text = remove_urls(text)
    text = remove_emails(text)
    text = remove_phone_numbers(text)
    text = remove_html_tags(text)
    text = remove_emojis(text)
    text = remove_hashtags_mentions(text)
    text = reduce_lengthening(text)
    text = remove_punctuation(text)
    text = remove_numbers(text)
    text = remove_extra_whitespace(text)

    print("\nCleaned Text:\n", text)

    words, sentences = tokenize_text(text)
    print("\nTokens:\n", words)

    words_no_stop = remove_stopwords(words)
    print("\nAfter Stopword Removal:\n", words_no_stop)

    stemmed = apply_stemming(words_no_stop)
    print("\nStemmed Words:\n", stemmed)

    lemmatized = apply_lemmatization(words_no_stop)
    print("\nLemmatized Words:\n", lemmatized)

    return {
        "clean_text": text,
        "tokens": words,
        "no_stopwords": words_no_stop,
        "stemmed": stemmed,
        "lemmatized": lemmatized
    }


In [12]:
# ==============================
# 9. Run Pipeline
# ==============================

output = preprocess_text(text)


Original Text:
 
OMG!!! This product is sooo goooood 😍😍!!! 
I bought it from https://example.com on 12/12/2024.
Contact me at test@email.com or call 9876543210.
#Amazing #BestProduct Ever!!!


Cleaned Text:
 omg this product is soo good i bought it from on contact me at or call ever

Tokens:
 ['omg', 'this', 'product', 'is', 'soo', 'good', 'i', 'bought', 'it', 'from', 'on', 'contact', 'me', 'at', 'or', 'call', 'ever']

After Stopword Removal:
 ['omg', 'product', 'soo', 'good', 'bought', 'contact', 'call', 'ever']

Stemmed Words:
 ['omg', 'product', 'soo', 'good', 'bought', 'contact', 'call', 'ever']

Lemmatized Words:
 ['omg', 'product', 'soo', 'good', 'bought', 'contact', 'call', 'ever']


In [13]:
output

{'clean_text': 'omg this product is soo good i bought it from on contact me at or call ever',
 'tokens': ['omg',
  'this',
  'product',
  'is',
  'soo',
  'good',
  'i',
  'bought',
  'it',
  'from',
  'on',
  'contact',
  'me',
  'at',
  'or',
  'call',
  'ever'],
 'no_stopwords': ['omg',
  'product',
  'soo',
  'good',
  'bought',
  'contact',
  'call',
  'ever'],
 'stemmed': ['omg',
  'product',
  'soo',
  'good',
  'bought',
  'contact',
  'call',
  'ever'],
 'lemmatized': ['omg',
  'product',
  'soo',
  'good',
  'bought',
  'contact',
  'call',
  'ever']}